In [1]:
!pip install "numpy>=2.0.0,<2.3.0" "opencv-python-headless<4.11" scipy scikit-image lightgbm xgboost seaborn rembg[gpu] onnxruntime-gpu pywavelets tqdm joblib seaborn lightgbm xgboost scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 904.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.0 MB/s eta 0:00:00


In [2]:
import os
import glob
import cv2
import numpy as np
import tarfile
import requests
import pywt
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed
from rembg import new_session, remove
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import skew

In [3]:
def setup_dataset():
    url = "https://download.microsoft.com/download/5/6/a/56a19dd6-35fb-47a2-a9b2-74e66d269345/msrcorid.tar.gz"
    archive_name = "msrcorid.tar.gz"
    extract_dir = "msrc_data"

    if os.path.exists(extract_dir):
        print(f"Dataset directory '{extract_dir}' already exists. Skipping download.")
        return extract_dir

    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(archive_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Extracting dataset...")
    with tarfile.open(archive_name, "r:gz") as tar:
        tar.extractall(path=extract_dir)
    return extract_dir

###Boundry Descriptors

In [4]:
def get_perimeter(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.arcLength(max(contours, key=cv2.contourArea), True) if contours else 0

def get_diameter(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        _, radius = cv2.minEnclosingCircle(cnt)
        return radius * 2
    return 0

def get_cog(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        M = cv2.moments(cnt)
        if M['m00'] != 0:
            cx = int(M['m10'] / M['m00'])
            cy = int(M['m01'] / M['m00'])
            return [cx, cy]
    return [0, 0]

def get_boundary_fourier(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea).reshape(-1, 2)
        complex_boundary = cnt[:, 0] + 1j * cnt[:, 1]
        fft_coeffs = np.abs(np.fft.fft(complex_boundary))
        if fft_coeffs[0] != 0:
            fft_coeffs = fft_coeffs / fft_coeffs[0]

        res = fft_coeffs[1:5].tolist()
        while len(res) < 4:
            res.append(0.0)
        return res
    return [0.0, 0.0, 0.0, 0.0]

def get_chain_code_and_shape_num(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        chain_code = []
        lookup = {(1, 0): 0, (1, 1): 1, (0, 1): 2, (-1, 1): 3,
                  (-1, 0): 4, (-1, -1): 5, (0, -1): 6, (1, -1): 7}
        for i in range(len(cnt) - 1):
            dx = np.sign(cnt[i+1][0][0] - cnt[i][0][0])
            dy = np.sign(cnt[i+1][0][1] - cnt[i][0][1])
            if (dx, dy) in lookup:
                chain_code.append(lookup[(dx, dy)])

        if chain_code:
            cc_mean = np.mean(chain_code)
            first_diff = [(chain_code[i] - chain_code[i-1]) % 8 for i in range(len(chain_code))]
            shape_num = np.mean(first_diff)
            return [cc_mean, shape_num]
    return [0, 0]

def get_curvature_mean(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        epsilon = 0.01 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        if len(approx) >= 3:
            angles = []
            for i in range(len(approx)):
                p1, p2, p3 = approx[i - 1][0], approx[i][0], approx[(i + 1) % len(approx)][0]
                v1, v2 = p1 - p2, p3 - p2
                cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)
                angles.append(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
            return np.mean(angles)
    return 0

###Region Descriptors

In [5]:
def get_area(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.contourArea(max(contours, key=cv2.contourArea)) if contours else 0

def get_circularity(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        return (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
    return 0

def get_compactness(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        return (perimeter ** 2) / area if area > 0 else 0
    return 0

def get_rectangularity(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        _, (w, h), _ = cv2.minAreaRect(cnt)
        rect_area = w * h
        return area / rect_area if rect_area > 0 else 0
    return 0

def get_eccentricity(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        if len(cnt) >= 5:
            _, (MA, ma), _ = cv2.fitEllipse(cnt)
            a, b = max(MA, ma) / 2.0, min(MA, ma) / 2.0
            return np.sqrt(1 - (b / a) ** 2) if a > 0 else 0
    return 0

def get_euler_number(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    if contours and hierarchy is not None:
        objects = sum(1 for i in range(len(contours)) if hierarchy[0][i][3] == -1)
        holes = sum(1 for i in range(len(contours)) if hierarchy[0][i][3] != -1)
        return objects - holes
    return 1

def get_hole_area_ratio(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    if contours and hierarchy is not None:
        total_obj_area = sum(cv2.contourArea(c) for i, c in enumerate(contours) if hierarchy[0][i][3] == -1)
        hole_area = sum(cv2.contourArea(c) for i, c in enumerate(contours) if hierarchy[0][i][3] != -1)
        return hole_area / total_obj_area if total_obj_area > 0 else 0
    return 0

def get_mpp_area(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        hull = cv2.convexHull(cnt)
        return cv2.contourArea(hull)
    return 0

###GLCM

In [6]:
def compute_base_glcm(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    return glcm

def get_contrast(img):
    glcm = compute_base_glcm(img)
    return graycoprops(glcm, 'contrast').mean()

def get_dissimilarity(img):
    glcm = compute_base_glcm(img)
    return graycoprops(glcm, 'dissimilarity').mean()

def get_homogeneity(img):
    glcm = compute_base_glcm(img)
    return graycoprops(glcm, 'homogeneity').mean()

def get_energy(img):
    glcm = compute_base_glcm(img)
    return graycoprops(glcm, 'energy').mean()

def get_correlation(img):
    glcm = compute_base_glcm(img)
    return graycoprops(glcm, 'correlation').mean()

def get_asm(img):
    glcm = compute_base_glcm(img)
    return graycoprops(glcm, 'ASM').mean()

def get_entropy(img):
    glcm = compute_base_glcm(img)
    epsilon = 1e-10
    entropy_list = []
    for i in range(4):
        matrix = glcm[:, :, 0, i]
        p = matrix[matrix > 0]
        entropy_list.append(-np.sum(p * np.log2(p + epsilon)))
    return np.mean(entropy_list)

###Spectral Features

In [7]:
def get_fourier_coefficients(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    f_transform = np.fft.fft2(gray)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = np.abs(f_shift)
    return [np.mean(magnitude_spectrum), np.max(magnitude_spectrum)]

def get_dct_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    gray_resized = cv2.resize(gray, (64, 64))
    img_float = np.float32(gray_resized) / 255.0
    dct = cv2.dct(img_float)
    low_freq = dct[0:5, 0:5].flatten()
    return low_freq[1:].tolist()

def get_wavelet_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    coeffs = pywt.dwt2(gray, 'haar')
    LL, (LH, HL, HH) = coeffs
    wav_features = []
    for band in [LH, HL, HH]:
        mean_val = np.mean(band)
        energy_val = np.mean(np.square(band))
        wav_features.extend([mean_val, energy_val])
    return wav_features

###Local Features

In [8]:
def get_harris_response(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    dst = cv2.cornerHarris(np.float32(gray), 2, 3, 0.04)
    return [np.mean(dst), np.sum(dst > 0.01 * dst.max())]

def get_sift_descriptors(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    sift = cv2.SIFT_create(nfeatures=128)
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    if descriptors is not None:
        return np.mean(descriptors, axis=0).tolist()
    return [0] * 128

###Color Features

In [9]:
def get_lab_color_stats(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    mean, std = cv2.meanStdDev(lab, mask=mask)

    return mean.flatten().tolist() + std.flatten().tolist()

def get_color_histogram_bins(img, bins=8):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    h_hist = cv2.calcHist([hsv], [0], mask, [bins], [0, 180])
    s_hist = cv2.calcHist([hsv], [1], mask, [bins], [0, 256])

    h_hist = cv2.normalize(h_hist, h_hist).flatten()
    s_hist = cv2.normalize(s_hist, s_hist).flatten()

    return np.concatenate([h_hist, s_hist]).tolist()

def get_color_moments(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    moments = []
    for i in range(3):
        channel = hsv[:, :, i][mask > 0]
        if len(channel) > 0:
            mu = np.mean(channel)
            sigma = np.std(channel)
            gamma = skew(channel)
            moments.extend([mu, sigma, gamma])
        else:
            moments.extend([0, 0, 0])
    return moments

###Extera Featurs

In [10]:
from skimage.feature import hog, local_binary_pattern

def get_hog_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    gray_res = cv2.resize(gray, (64, 64))
    fd = hog(gray_res, orientations=9, pixels_per_cell=(8, 8),
             cells_per_block=(2, 2), visualize=False)
    return fd.tolist()

def get_lbp_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    radius = 1
    n_points = 8 * radius
    lbp = local_binary_pattern(gray, n_points, radius, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    return hist.tolist()

In [11]:
def process_image_pipeline(image_path, session):
    try:
        img = cv2.imread(image_path)
        if img is None: return None

        # 1. CLAHE Pre-processing (From File 1)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_clahe = clahe.apply(l)
        img_clahe = cv2.cvtColor(cv2.merge((l_clahe, a, b)), cv2.COLOR_LAB2BGR)

        # 2. Advanced Denoising
        img_denoised = cv2.fastNlMeansDenoisingColored(img_clahe, None, 10, 10, 7, 21)

        # 3. GPU Segmentation (Background Removal)
        result_rgba = remove(img_denoised, session=session)

        if result_rgba.shape[2] == 4:
            alpha = result_rgba[:, :, 3]
            _, mask = cv2.threshold(alpha, 10, 255, cv2.THRESH_BINARY)
            if np.sum(mask) == 0: return None
            segmented_img = cv2.bitwise_or(img_denoised, img_denoised, mask=mask)

            # 4. Feature Extraction
            vec = []

            # Shape & Boundary
            vec.append(get_perimeter(segmented_img))
            vec.append(get_diameter(segmented_img))
            vec.extend(get_cog(segmented_img))
            vec.extend(get_boundary_fourier(segmented_img))
            vec.extend(get_chain_code_and_shape_num(segmented_img))
            vec.append(get_curvature_mean(segmented_img))
            vec.append(get_area(segmented_img))
            vec.append(get_circularity(segmented_img))
            vec.append(get_compactness(segmented_img))
            vec.append(get_rectangularity(segmented_img))
            vec.append(get_eccentricity(segmented_img))
            vec.append(get_euler_number(segmented_img))
            vec.append(get_hole_area_ratio(segmented_img))
            vec.append(get_mpp_area(segmented_img))

            # Texture (GLCM)
            vec.append(get_contrast(segmented_img))
            vec.append(get_dissimilarity(segmented_img))
            vec.append(get_homogeneity(segmented_img))
            vec.append(get_energy(segmented_img))
            vec.append(get_correlation(segmented_img))
            vec.append(get_asm(segmented_img))
            vec.append(get_entropy(segmented_img))

            # Spectral & Local Features (Optimized SIFT)
            vec.extend(get_fourier_coefficients(segmented_img))
            vec.extend(get_dct_features(segmented_img))
            vec.extend(get_wavelet_features(segmented_img))
            vec.extend(get_harris_response(segmented_img))
            vec.extend(get_sift_descriptors(segmented_img)) # Now Optimized

            # Color & Structural (HOG/LBP)
            vec.extend(get_lab_color_stats(segmented_img)) # Bug Fixed
            vec.extend(get_color_histogram_bins(segmented_img))
            vec.extend(get_color_moments(segmented_img))
            vec.extend(get_lbp_features(segmented_img))
            vec.extend(get_hog_features(segmented_img))

            return vec
    except Exception as e:
        # Silent fail or logging
        return None
    return None

In [12]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

if __name__ == "__main__":
    dataset_root = setup_dataset()
    dataset_path = os.path.join(dataset_root, "msrcorid")
    if not os.path.exists(dataset_path):
        dataset_path = dataset_root

    try:
        session = new_session("u2net", providers=['CUDAExecutionProvider'])
    except:
        session = new_session("u2net")

    classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    all_files, all_labels = [], []
    for cls in classes:
        files = glob.glob(os.path.join(dataset_path, cls, "**", "*.JPG"), recursive=True)
        all_files.extend(files)
        all_labels.extend([cls] * len(files))

    X_data, y_data = [], []
    for img_path, label in tqdm(zip(all_files, all_labels), total=len(all_files)):
        features = process_image_pipeline(img_path, session)
        if features:
            X_data.append(features)
            y_data.append(label)

    if len(X_data) > 0:
        X = np.array(X_data, dtype=object)
        X = np.array([np.array(i) for i in X])
        X = np.nan_to_num(X)

        y = np.array(y_data)
        le = LabelEncoder()
        y_encoded = le.fit_transform(y)

        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full, y_train_full, test_size=0.1875, random_state=42, stratify=y_train_full
        )

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        base_models = [
            ('lgbm', LGBMClassifier(n_estimators=1000, learning_rate=0.01, num_leaves=50,
                                    class_weight='balanced', random_state=42, n_jobs=-1)),
            ('rf', RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1)),
            ('xgb', XGBClassifier(n_estimators=500, learning_rate=0.05, eval_metric='mlogloss', random_state=42))
        ]

        stack_clf = StackingClassifier(
            estimators=base_models,
            final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced'),
            cv=5, n_jobs=-1
        )

        stack_clf.fit(X_train_scaled, y_train)

        y_pred = stack_clf.predict(X_test_scaled)
        y_prob = stack_clf.predict_proba(X_test_scaled)

        print("\n--- Evaluation Results ---")
        print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

        auc_score = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
        print(f"Final Model AUC: {auc_score:.4f}")

        plt.figure(figsize=(12, 10))
        sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='magma',
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title('Ensemble Stacking Confusion Matrix')
        plt.show()

    else:
        print("Error: Pipeline returned empty X_data.")

Extracting dataset...


/tmp/ipython-input-3098853341.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)
100%|███████████████████████████████████████| 176M/176M [00:00<00:00, 71.4GB/s]
  0%|          | 7/4323 [00:44<7:40:45,  6.41s/it]


KeyboardInterrupt: 